# 🎤 Soprano Danish TTS Fine-tuning with Unsloth

This notebook fine-tunes the Soprano TTS model on Danish speech data using Unsloth.

**Requirements:**
- Google Colab with GPU (T4 free tier works!)
- ~8GB VRAM minimum

**What we'll do:**
1. Install Unsloth (2-5x faster training)
2. Load Soprano model (Qwen3-based TTS)
3. Prepare Danish dataset
4. Fine-tune with LoRA
5. Test the model

## 1. Setup & Installation

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
%%capture
# Install Unsloth (takes ~2 minutes)
!pip install --no-deps trl peft accelerate bitsandbytes triton
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers

# Additional dependencies
!pip install datasets huggingface_hub soundfile librosa

In [ ]:
# Verify installation
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

from unsloth import FastLanguageModel
print("Unsloth loaded successfully!")

## 2. Load Soprano Model

In [ ]:
# Configuration
MODEL_NAME = "ekwek/Soprano-1.1-80M"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = False  # Full precision for better TTS quality

# LoRA configuration
LORA_R = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.0

In [ ]:
# Load model with Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-detect (bfloat16 on Ampere+, float16 otherwise)
    load_in_4bit=LOAD_IN_4BIT,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters():,}")

In [ ]:
# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",  # 3x less memory
    random_state=42,
)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")

## 3. Prepare Dataset

Option A: Upload your CORAL Danish dataset
Option B: Use a sample dataset for testing

In [ ]:
# Option A: Upload your dataset
# Uncomment and run this cell, then upload your train.json file

# from google.colab import files
# uploaded = files.upload()  # Upload train.json

In [ ]:
# Option B: Create a small test dataset
# This simulates the Soprano format for testing

TEST_MODE = True  # Set to False when using real data

if TEST_MODE:
    # Create dummy data in Soprano format
    # Real data should have audio tokens encoded
    test_data = [
        {"text": "<|audio|>Hej, jeg hedder Anna og jeg kommer fra København.<|endoftext|>"},
        {"text": "<|audio|>Velkommen til denne demonstration af dansk tale syntese.<|endoftext|>"},
        {"text": "<|audio|>Vejret i dag er solrigt med temperaturer omkring tyve grader.<|endoftext|>"},
    ] * 100  # Repeat for testing
    
    import json
    with open("train.json", "w") as f:
        json.dump(test_data, f)
    print(f"Created test dataset with {len(test_data)} samples")

In [ ]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset("json", data_files="train.json", split="train")
print(f"Loaded {len(dataset)} samples")
print(f"Sample: {dataset[0]}")

In [ ]:
# Format dataset for training
def format_for_training(example):
    """Format example for Soprano TTS training."""
    return {"text": example["text"]}

dataset = dataset.map(format_for_training)
dataset = dataset.shuffle(seed=42)

# Split into train/val
dataset = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = dataset["train"]
val_dataset = dataset["test"]

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

## 4. Training Configuration

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

# Training configuration
training_args = TrainingArguments(
    output_dir="./soprano-danish-unsloth",
    
    # Batch size (adjust based on GPU memory)
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # Effective batch = 8
    
    # Training duration
    max_steps=500,  # Increase for real training (e.g., 2500)
    # num_train_epochs=3,  # Alternative to max_steps
    
    # Learning rate
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    
    # Precision
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    
    # Optimizer
    optim="adamw_8bit",
    weight_decay=0.01,
    
    # Logging
    logging_steps=10,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    
    # Misc
    seed=42,
    report_to="none",  # or "wandb"
)

In [ ]:
# Create trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,  # Pack multiple samples for efficiency
    dataset_text_field="text",
)

print("Trainer ready!")

## 5. Train! 🚀

In [ ]:
# Show GPU memory before training
gpu_stats = torch.cuda.get_device_properties(0)
reserved = torch.cuda.memory_reserved(0) / 1024**3
print(f"GPU: {gpu_stats.name}")
print(f"Total memory: {gpu_stats.total_memory / 1024**3:.1f} GB")
print(f"Reserved before training: {reserved:.1f} GB")

In [ ]:
# Train!
print("Starting training...")
trainer.train()

In [ ]:
# Show GPU memory after training
reserved = torch.cuda.memory_reserved(0) / 1024**3
max_reserved = torch.cuda.max_memory_reserved(0) / 1024**3
print(f"Memory used during training: {max_reserved:.1f} GB")

## 6. Save Model

In [ ]:
# Save LoRA adapters
model.save_pretrained("soprano-danish-lora")
tokenizer.save_pretrained("soprano-danish-lora")
print("LoRA adapters saved to 'soprano-danish-lora'")

In [ ]:
# Optional: Merge LoRA into base model and save
# This creates a standalone model without needing the adapter

# model.save_pretrained_merged("soprano-danish-merged", tokenizer)
# print("Merged model saved to 'soprano-danish-merged'")

In [ ]:
# Download the model
from google.colab import files

# Zip and download
!zip -r soprano-danish-lora.zip soprano-danish-lora/
files.download("soprano-danish-lora.zip")

## 7. Test the Model (Optional)

In [ ]:
# Quick inference test
FastLanguageModel.for_inference(model)

# Test generation
test_text = "Hej, dette er en test af den danske stemme."
inputs = tokenizer(test_text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
    )

generated = tokenizer.decode(outputs[0], skip_special_tokens=False)
print(f"Generated: {generated[:500]}...")

## 8. Push to Hugging Face Hub (Optional)

In [ ]:
# Login to Hugging Face
# from huggingface_hub import login
# login(token="your_token_here")

# Push to hub
# model.push_to_hub("your-username/soprano-danish-lora")
# tokenizer.push_to_hub("your-username/soprano-danish-lora")

---

## Notes

**For real training with CORAL data:**
1. Upload your `train.json` with proper audio tokens
2. Set `TEST_MODE = False`
3. Increase `max_steps` to 2500+
4. Consider filtering to single speaker (female) for best results

**Memory tips:**
- T4 (16GB): batch_size=2, grad_accum=4
- A100 (40GB): batch_size=8, grad_accum=2
- If OOM: reduce batch_size or enable 4-bit (LOAD_IN_4BIT=True)

**Resources:**
- [Unsloth Docs](https://docs.unsloth.ai/)
- [Soprano Model](https://huggingface.co/ekwek/Soprano-1.1-80M)